In [ ]:
%reload_ext autoreload
%autoreload 2

import os
import json
import plot_utils

QUERY_LOG_PATH = '../query-log/'

DATA_FOLDER = 'parachute-data'
DATA_NAME = 'imdb'
WORKLOAD_NAME = 'job'
LIST_NUM_THREADS = [1]

MACHINES = ['hausberg', 'heimgarten']

full_reduction_value = 0
import utils
json_data = utils.read_json(f'{WORKLOAD_NAME}-full-reduction.json')
for key in json_data:
  full_reduction_value += json_data[key]
print(f'full={full_reduction_value}')

df = plot_utils.collect_data(QUERY_LOG_PATH, DATA_NAME, WORKLOAD_NAME, MACHINES, LIST_NUM_THREADS,
  ['duckdb', 'rpt'])
df.to_csv(f'{WORKLOAD_NAME}-latex-test.csv', index=False)
df

In [ ]:
__configs__ = [
  # 'sip',
  'rpt',

  # 'parachute-2-0-1-0',
  # 'parachute-4-0-1-0',
  # 'parachute-8-0-1-0',

  # parachute[duckdb].
  # 'parachute-2-0-1-2-0',
  # 'parachute-4-0-1-2-0',
  # 'parachute-8-0-1-2-0',
  # 'parachute-16-0-1-2-0',

  # NOTE: This 4 were valid.
  # 'parachute-2-0-1-3-0',
  # 'parachute-4-0-1-3-0',
  # 'parachute-8-0-1-3-0',
  # 'parachute-16-0-1-3-0',
  # ---- #

  # parachute[duckdb + sip].
  # NOTE: This 4 were valid.
  # 'sip_parachute-2-0-1-2-0',
  # 'sip_parachute-4-0-1-2-0',
  # 'sip_parachute-8-0-1-2-0',
  # 'sip_parachute-16-0-1-2-0',
  # ---- #

  # 'sip_parachute-16-0-1-3-0',
  # 'sip_parachute-2-0-1-2-1',
  # 'sip_parachute-2-0-1-3-1',
  # 'sip_parachute-4-0-1-2-1',
  # 'sip_parachute-4-0-1-3-1',
  # 'sip_parachute-8-0-1-2-1',
  # 'sip_parachute-8-0-1-3-1',
  # 'sip_parachute-16-0-1-2-1',
  # 'sip_parachute-16-0-1-3-1',
  # 'sip_parachute-16-0-1-0',
  # 'sip_parachute-16-0-1-1',
  # 'sip_parachute-16-0-1-2',
]

def get_config_color(config):
  if config.startswith('sip_parachute'):
    if config.endswith('2-0'):
      return '#CC0000' # red.
  elif config.startswith('parachute'):
    if config.endswith('2-0'):
      return '#964B00' # brown
    if config.endswith('3-0'):
      return '#964B00'
      # return '#CC0000' # red.
  return 'black'

In [ ]:
df[(df['config'] == 'duckdb') & (df['mode'] == 'hot') & (df['query'] == '1a_1a362')]

In [ ]:
df[(df['config'] == 'sip') & (df['mode'] == 'hot') & (df['query'] == '1a_1a362')]

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

matplotlib.rcParams.update({
  'pgf.texsystem': 'pdflatex',
  'font.family': 'serif',
  'text.usetex': True,
  'pgf.rcfonts': False,
})

FIG_WIDTH, FIG_HEIGHT = 5, 2

def get_fontsize(msg):
  if msg == 'legend':
    return 10
  if msg == 'x-ticks':
    return 10
  if msg == 'y-ticks':
    return 10
  return 11

def load_space_consumption():
  # Load space consumption stats from JSON files in the stats directory.
  space_data = {}

  for file in os.listdir(os.path.join(DATA_FOLDER, DATA_NAME)):
    if WORKLOAD_NAME == 'ceb':
      if file.startswith(f'{WORKLOAD_NAME}_parachute-size-'):
        parts = file.replace('.json', '').split('-')
        db_file = f'{WORKLOAD_NAME}_{DATA_NAME}-parachute-{parts[3]}-{parts[4]}-{parts[5]}'

        with open(os.path.join(DATA_FOLDER, DATA_NAME, file), 'r') as f:
          stats = json.load(f)
          space_consumption = ((stats['final_db_file_size'] - stats['init_db_file_size']) / stats['init_db_file_size']) * 100
          space_data[db_file] = space_consumption
    elif WORKLOAD_NAME == 'job':
      if file.startswith(f'parachute-size-'):
        parts = file.replace('.json', '').split('-')
        db_file = f'{DATA_NAME}-parachute-{parts[3]}-{parts[4]}-{parts[5]}'

        with open(os.path.join(DATA_FOLDER, DATA_NAME, file), 'r') as f:
          stats = json.load(f)
          space_consumption = ((stats['final_db_file_size'] - stats['init_db_file_size']) / stats['init_db_file_size']) * 100
          space_data[db_file] = space_consumption

  return space_data

def plot_metric_vs_space(axs, df, space_data, y_metric, y_label, mode, num_threads, selected_configs=None):
  # Setup the figure.
  fig = None
  if axs is None:
    fig, axs = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT))
  else:
    fig = None  

  # Filter.
  filtered_df = df[(df['mode'] == mode) & (df['num-threads'] == num_threads)].copy()

  # Take my configs.
  my_configs = None
  if selected_configs is None:
    my_configs = filtered_df['config'].unique().tolist()
  else:
    my_configs = selected_configs.copy()
  assert my_configs is not None

  print(my_configs)

  # Filter based on the configs.
  my_configs = ['duckdb'] + my_configs
  filtered_df = filtered_df[filtered_df['config'].isin(my_configs)]

  # Map space consumption data
  filtered_df['space-consumption'] = filtered_df['db-file'].map(space_data).fillna(0)

  # Aggregate by (competitor_type, config, mode, num_th)
  agg_df = filtered_df.groupby(['config', 'mode', 'num-threads'], as_index=False).agg(
    {y_metric: 'sum', 'base-size': 'sum', 'space-consumption': 'first'}
  )

  # Compute the metric for a given `config`.
  def compute_metric(config):
    if config in my_configs:
      return agg_df.loc[agg_df['config'] == config, y_metric].values[0]
    return None

  # Get DuckDB base-size for the metric
  base_size = agg_df.loc[agg_df['config'] == 'duckdb', 'base-size'].values[0]
  duckdb_metric = compute_metric('duckdb')
  sip_metric = compute_metric('sip')
  rpt_metric = compute_metric('rpt')

  # Normalize values by dividing them by the DuckDB value (scaled relative to DuckDB)
  if y_metric == '#made-it':
    # TODO: Why don't we also have for SIP?
    # TODO: Is this actually correct?
    agg_df[y_metric] = (agg_df[y_metric] - full_reduction_value) / (base_size - full_reduction_value) * 100  # Scale everything relative to DuckDB
  elif y_metric == 'latency (s)':
    agg_df[y_metric] = agg_df[y_metric] / duckdb_metric
    sip_metric = sip_metric / duckdb_metric if sip_metric is not None else None
    rpt_metric = rpt_metric / duckdb_metric if rpt_metric is not None else None

  # First, handle general configurations
  for config, group in agg_df.groupby('config'):
    if config in ['duckdb', 'sip', 'rpt']:
      continue
    axs.scatter(group['space-consumption'], group[y_metric], label=config, s=6, color=get_config_color(config))
    axs.plot(group['space-consumption'], group[y_metric])

  # DuckDB and SIP baselines
  for info in [
    {'tag': 'duckdb', 'label': r'\texttt{duckdb}', 'color': 'black', 'linestyle': '-',
      'pure' : 'duckdb',
        # JOB: Made-it
        'job-#made-it-x-shift' : 2.0,
        'job-#made-it-y-shift' : 1.8,
        # JOB: Latency.
        'job-latency (s)-x-shift' : 2.0,
        'job-latency (s)-y-shift' : 0.03,

        # CEB: Made-it
        'ceb-#made-it-x-shift' : 2.0,
        'ceb-#made-it-y-shift' : 1.8,
        # CEB: Latency.
        'ceb-latency (s)-x-shift' : 2.0,
        'ceb-latency (s)-y-shift' : 0.03
    },
    {'tag': 'sip', 'label': r'\texttt{duckdb\,+\,sip}', 'color': '#00008B', 'linestyle': '-',
      'pure': 'duckdb + sip',
        # JOB: Made-it.
        'job-#made-it-x-shift' : 3.4,
        'job-#made-it-y-shift' : 1.25,
        # JOB: Latency.
        'job-latency (s)-x-shift' : 3.4,
        'job-latency (s)-y-shift' : -0.04,

        # CEB: Made-it.
        'ceb-#made-it-x-shift' : 3.4,
        'ceb-#made-it-y-shift' : 1.25,
        # CEB: Latency.
        'ceb-latency (s)-x-shift' : 3.4,
        'ceb-latency (s)-y-shift' : -0.04
    },
    {'tag': 'rpt', 'label': r'\texttt{duckdb\,+\,rpt}', 'color': '#00008B', 'linestyle': '-',
      'pure': 'duckdb + rpt',
        # JOB: Made-it.
        'job-#made-it-x-shift' : 3.4,
        'job-#made-it-y-shift' : 1.25,
        # JOB: Latency.
        'job-latency (s)-x-shift' : 3.4,
        'job-latency (s)-y-shift' : -0.04,

        # CEB: Made-it.
        'ceb-#made-it-x-shift' : 3.4,
        'ceb-#made-it-y-shift' : 1.25,
        # CEB: Latency.
        'ceb-latency (s)-x-shift' : 3.4,
        'ceb-latency (s)-y-shift' : -0.04
    }
  ]:
    if info['tag'] not in agg_df['config'].values:
      continue
    y_value = agg_df.loc[agg_df['config'] == info['tag'], y_metric].values[0]

    axs.axhline(y=y_value, color=info['color'], linestyle=info['linestyle'], label=info['label'], lw=3)
    axs.text(
      axs.get_xlim()[1] - info[f'{WORKLOAD_NAME}-{y_metric}-x-shift'],
      y_value + info[f'{WORKLOAD_NAME}-{y_metric}-y-shift'],
      info['label'],
      color='black',
      va='center',
      ha='left',
      fontsize=get_fontsize('legend'),
      fontweight='bold'
    )

  # Handle parachute configurations separately to draw lines between specific points
  parachute_configs = [config for config in agg_df['config'] if 'parachute' in config]
  parachute_configs.sort(key=lambda x: (int(x.split('-')[1]), int(x.split('-')[2]), int(x.split('-')[3]), int(x.split('-')[4]), int(x.split('-')[5])))  # Sort by pbw, transitivity, adaptivity, altitude, opt
 
  for competitor_type, competitor_label in [
    ('parachute', r'\texttt{parachute[duckdb]}'),
    ('sip_parachute', r'\texttt{parachute[duckdb\,+\,sip]}'),
    ('rpt_parachute', r'\texttt{parachute[duckdb\,+\,rpt]}')
  ]:
    for type in ['2-0', '3-0']:
      parachute_points = []
      for config in parachute_configs:
        if not config.endswith(type):
          continue
        if not config.startswith(competitor_type):
          continue

        group = agg_df[agg_df['config'] == config]
        parachute_points.append(group[['space-consumption', y_metric]].values[0])

      # Skip if empty.
      if not parachute_points:
        continue

      # Connect with SIP or DuckDB (only for #non-dangling).
      connecting_value = None
      if competitor_type == 'parachute':
        connecting_value = agg_df.loc[agg_df['config'] == 'duckdb', y_metric].values[0]
      elif competitor_type == 'sip_parachute':
        connecting_value = agg_df.loc[agg_df['config'] == 'sip', y_metric].values[0]
      else:
        assert 0

      assert connecting_value is not None
      parachute_points = [[0, connecting_value]] + parachute_points

      # Now plot lines connecting parachute points
      parachute_points = np.array(parachute_points)
      line, = axs.plot(
        parachute_points[:, 0],
        parachute_points[:, 1],
        linestyle='-',
        color=get_config_color(competitor_type + '|' + type),
        lw=2.5
      )

      # Calculate slope using the last two points
      x1, y1 = parachute_points[-2]
      x2, y2 = parachute_points[-1]
      angle = np.rad2deg(np.arctan2(y2 - y1, x2 - x1))

      # Add label on top of the parachute line (in the middle) with slope-based rotation
      # https://stackoverflow.com/questions/18780198/how-to-rotate-matplotlib-annotation-to-match-a-line
      mid_x = np.mean(parachute_points[-2:][:, 0])
      mid_y = np.mean(parachute_points[-2:][:, 1])

      special_shifts = {
        'job' : {
          '#made-it' : -0.5,
          'latency (s)' : 0.00,
        },
        'ceb' : {
          '#made-it' : -0.5,
          'latency (s)' : 0.00,
        }
      }

      if not competitor_type.startswith('sip_'):
        y_shift = special_shifts[WORKLOAD_NAME][y_metric]
      else:
        y_shift = -0.075
      
      axs.text(
        mid_x,
        mid_y + y_shift,
        competitor_label,
        ha='center',
        va='bottom', 
        color='black',
        fontsize=get_fontsize('legend'),
        fontweight='bold',
        transform_rotates_text=True,
        rotation=angle
      )

  # https://stackoverflow.com/questions/11744990/how-to-set-auto-for-upper-limit-but-keep-a-fixed-lower-limit
  axs.set_xlim(left=0)
  
  # Format x-tick labels as percentages
  if y_metric == '#made-it':
    if WORKLOAD_NAME == 'job':
      axs.set_yticks(sorted([0, 5, 10, 15, 20, 25]))
      axs.set_yticklabels([rf'{int(y)}\%' for y in axs.get_yticks()], fontsize=get_fontsize('y-ticks'))
    elif WORKLOAD_NAME == 'ceb':
      axs.set_yticks(sorted([0, 5, 10, 20, 30, 40, 50]))
      axs.set_yticklabels([rf'{int(y)}\%' for y in axs.get_yticks()], fontsize=get_fontsize('y-ticks'))
  elif y_metric == 'latency (s)':
    rounded_sip = round(sip_metric, 1) if sip_metric is not None else None
    rounded_rpt = round(rpt_metric, 1) if rpt_metric is not None else None
    axs.set_yticks(sorted([1.0, 0.9, 0.8, 0.7, 0.6, 0.5]))
    axs.set_yticklabels([y for y in axs.get_yticks()], fontsize=get_fontsize('y-ticks'))

  # Set the y-lim.
  if y_metric == '#made-it':
    axs.set_ylim(0, max(agg_df[y_metric]) * 1.15)
  elif y_metric == 'latency (s)':
    axs.set_ylim(top=max(agg_df[y_metric]) * 1.1)
    # axs.set_ylim(0.9, max(agg_df[y_metric]) * 1.05)

  # Set x-ticks first
  axs.set_xticks([0, 5, 10, 15])

  # Format x-tick labels as percentages
  axs.set_xticklabels([rf'{int(x)}\%' for x in axs.get_xticks()], fontsize=get_fontsize('x-ticks'))
  
  # Set labels.
  axs.set_xlabel(r'Extra disk space', fontsize=get_fontsize('xlabel'))
  axs.set_ylabel(y_label, fontsize=get_fontsize('ylabel'))

  # Put grid lines.
  # axs.grid(axis='y')
  axs.grid(True, axis='y', color='gray', linestyle='-', linewidth=0.5, alpha=0.5)

  # Save fig.
  if fig is not None:
    fig.savefig(
      f"plots/{WORKLOAD_NAME}-metric-{y_metric.replace('#', '')}-plot.pdf",
      format="pdf",
      bbox_inches="tight",
      pad_inches=0.01,
      dpi=300
    )

  # plt.tight_layout()
  if fig is not None:
    plt.show()

# Load space data.
space_data = load_space_consumption()

# Single-threaded.
plot_metric_vs_space(None, df, space_data, y_metric='#made-it', y_label=r'Dangling tuples', mode='hot', num_threads=1, selected_configs=__configs__)
# plot_metric_vs_space(None, df, space_data, y_metric='cout', y_label=r'Total Cout', mode='hot', num_threads=1, selected_configs=__configs__)
# plot_metric_vs_space(None, df, space_data, y_metric='latency (s)', y_label=r'Normalized exec.~time', mode='hot', num_threads=1, selected_configs=__configs__)

# 48 threads.
# plot_metric_vs_space(df, space_data, y_metric='#made-it', y_label=r'Total #made-it', mode='hot', num_threads=48, selected_configs=__configs__)
# plot_metric_vs_space(df, space_data, y_metric='cout', y_label=r'Total Cout', mode='hot', num_threads=48, selected_configs=__configs__)
# plot_metric_vs_space(df, space_data, y_metric='latency (s)', y_label='Total latency (s)', mode='hot', num_threads=48, selected_configs=__configs__)

In [ ]:
def plot_full(df, space_data, mode, num_threads, selected_configs):
  # Set up the figure.
  fig, axs = plt.subplots(1, 2, figsize=(FIG_WIDTH * 2, FIG_HEIGHT), gridspec_kw={'wspace': 0.2})

  # Single-threaded.
  plot_metric_vs_space(axs[0], df, space_data, y_metric='#made-it', y_label=r'Dangling tuples', mode=mode, num_threads=num_threads, selected_configs=selected_configs)
  plot_metric_vs_space(axs[1], df, space_data, y_metric='latency (s)', y_label=r'Normalized exec.~time', mode=mode, num_threads=num_threads, selected_configs=selected_configs)

  # Save.
  fig.savefig(
    f"latex-plots/{WORKLOAD_NAME}-plot.pdf",
    format="pdf",
    bbox_inches="tight",
    pad_inches=0.01,
    dpi=300
  )

  plt.show()
  
plot_full(df, space_data, mode='hot', num_threads=1, selected_configs=__configs__)